[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-08-work-pools-workers.ipynb#scrollTo=a1b2c3d4)

---
# Day 8 · Work Pools and Workers — Local, Docker, and Subprocess
**certified-journeys / prefect-certified** &nbsp;|&nbsp; Practice

> **Goal for today:** Understand how work pools decouple scheduling from execution, create a local process pool, configure a Docker pool, and trace how workers poll the queue to pick up and run scheduled flow runs.

---
## Architecture overview

The Prefect execution model has three distinct layers:

```
Prefect Server / Cloud
       │  (schedules runs, stores state)
       ▼
  Work Pool  ←── deployment targets this pool
       │  (queue of pending runs)
       ▼
    Worker  ←── polls the pool on an interval
       │  (fetches run, spins up infrastructure)
       ▼
  Flow Run  ←── subprocess / container / pod
```

| Component | Responsibility | Lives on |
|---|---|---|
| Prefect Server | Schedule, state, logs | Cloud or self-hosted |
| Work Pool | Queue of runs waiting for a worker | Server-side |
| Worker | Long-running process that polls the pool | Your infrastructure |
| Flow Run | Actual execution of your code | Subprocess / container |

This architecture means your server never needs to reach your infrastructure — **workers pull work, they are not pushed to.**

In [ ]:
%pip install -q "prefect>=2.14" docker

---
## Step 1 · Work pool types and when to use them

Prefect ships several pool types out of the box. The type determines what infrastructure each worker
spawns when it picks up a run.

| Pool type | Spawns | Best for |
|---|---|---|
| `process` | Local subprocess | Development, single-machine pipelines |
| `docker` | Docker container | Reproducible, isolated environments |
| `kubernetes` | Kubernetes pod | Production auto-scaling |
| `ecs` | AWS ECS task | AWS-native deployments |
| `vertex-ai` | Vertex AI custom job | GCP ML workloads |
| `cloud-run` | GCP Cloud Run job | Serverless on GCP |

**Process pools** are the best starting point — zero extra infrastructure, immediate feedback.
When you switch to Docker or Kubernetes later you only change the pool type; your flow code is untouched.

CLI to list available pool types:
```bash
prefect work-pool types
```

In [ ]:
import os
import prefect

# Use the ephemeral in-process server so no server process is needed in Colab
os.environ["PREFECT_API_URL"] = ""
os.environ["PREFECT_SERVER_ANALYTICS_ENABLED"] = "false"

print(f"Prefect version: {prefect.__version__}")
print("Using in-process ephemeral server (PREFECT_API_URL is empty)")

# Show all built-in worker types via the registry
from prefect.workers.base import BaseWorker

print("\nBuilt-in worker base class:", BaseWorker.__name__)
print("Production command to create a process pool:")
print("  prefect work-pool create --type process my-pool")
print("\nProduction command to start a worker:")
print("  prefect worker start --pool my-pool")

**What just happened?**
- `PREFECT_API_URL=""` activates the **ephemeral server** — Prefect runs an in-process orchestration layer, no socket needed
- In production you export `PREFECT_API_URL=https://api.prefect.cloud/api/accounts/.../workspaces/...`
- **Workers pull, not push** — the server never initiates a connection to your infrastructure; this makes firewalled environments easy to support

---
## Step 2 · Create a process work pool via the Python client

In day-to-day use you create work pools once with the CLI:
```bash
prefect work-pool create --type process my-pool
```

The Python client API does the same thing programmatically — useful for automated setup scripts
and CI/CD pipelines.

**Work pool fields:**

| Field | Description |
|---|---|
| `name` | Unique pool identifier — deployments reference this name |
| `type` | Infrastructure type (`process`, `docker`, `kubernetes`, …) |
| `base_job_template` | JSON schema of infra settings (image, CPU, memory, env vars) |
| `is_paused` | When `True`, workers will not pick up runs from this pool |
| `concurrency_limit` | Max simultaneous runs; `None` = unlimited |
| `description` | Free-text description visible in the UI |

In [ ]:
import asyncio
import json
from prefect.client.orchestration import get_client
from prefect.client.schemas.actions import WorkPoolCreate

PROCESS_POOL_NAME = "my-pool"

async def create_process_pool():
    async with get_client() as client:
        try:
            pool = await client.create_work_pool(
                WorkPoolCreate(
                    name=PROCESS_POOL_NAME,
                    type="process",
                    description="Local process pool for development — runs flows as subprocesses.",
                    # concurrency_limit=None means unlimited concurrent runs
                )
            )
            print(f"Work pool created successfully!")
            print(f"  Name:        {pool.name}")
            print(f"  Type:        {pool.type}")
            print(f"  Description: {pool.description}")
            print(f"  Is paused:   {pool.is_paused}")
            print(f"  Concurrency: {pool.concurrency_limit} (None = unlimited)")
            return pool
        except Exception as e:
            print(f"Pool creation note: {e}")
            return None

pool = asyncio.run(create_process_pool())

print("\nCLI equivalent:")
print(f"  prefect work-pool create --type process {PROCESS_POOL_NAME}")
print(f"  prefect work-pool inspect {PROCESS_POOL_NAME}")

**What just happened?**
- `WorkPoolCreate` sends the pool configuration to the Prefect server — the pool now exists in the database
- `type="process"` means each run will be executed as a **subprocess** on the worker's machine
- `is_paused=False` (default) — the pool is immediately ready to accept work
- **Concurrency limit** controls burst protection: setting `concurrency_limit=4` prevents more than 4 simultaneous runs from this pool

---
## Step 3 · Deploy a flow to the process pool

A deployment connects a flow to a work pool. The deployment manifest answers:
- **What** to run: `entrypoint` — `file.py:function`
- **Where** to run: `work_pool_name`
- **With what defaults**: `parameters`
- **When**: `schedule` (covered in Day 9)

```bash
# CLI deployment (production GitOps style)
prefect deploy my_flow.py:my_flow \
  --name etl-daily \
  --pool my-pool \
  --param source=api-v1
```

Below we use the Python client — the same data is sent to the server either way.

In [ ]:
import asyncio
from prefect import flow, task, get_run_logger
from prefect.testing.utilities import prefect_test_harness

# Define a realistic ETL flow we will deploy to the pool
@task(retries=2, retry_delay_seconds=3)
def extract(source: str, batch_size: int) -> list[dict]:
    logger = get_run_logger()
    logger.info(f"Extracting {batch_size} records from '{source}'")
    # Simulate API response — replace with real client in production
    return [{"id": i, "source": source, "value": i * 1.5} for i in range(1, batch_size + 1)]

@task
def transform(records: list[dict], multiplier: float) -> list[dict]:
    return [{**r, "value": round(r["value"] * multiplier, 4)} for r in records]

@task
def load(records: list[dict], destination: str) -> int:
    logger = get_run_logger()
    logger.info(f"Loading {len(records)} records to '{destination}'")
    # Simulate write — replace with database insert in production
    return len(records)

@flow(name="etl-pipeline", log_prints=True)
def etl_pipeline(
    source: str = "api-v1",
    destination: str = "warehouse",
    batch_size: int = 50,
    multiplier: float = 1.0,
) -> dict:
    """Extract → Transform → Load pipeline deployable to any work pool."""
    records = extract(source=source, batch_size=batch_size)
    transformed = transform(records=records, multiplier=multiplier)
    count = load(records=transformed, destination=destination)
    summary = {"source": source, "destination": destination, "loaded": count}
    print(f"ETL complete: {summary}")
    return summary

# Test the flow locally — same code that will run inside the worker's subprocess
with prefect_test_harness():
    result = etl_pipeline(source="test-api", batch_size=5, multiplier=2.0)
    print(f"\nLocal test result: {result}")

**What just happened?**
- `prefect_test_harness()` runs the flow with full Prefect tracking in an isolated in-process environment
- Every `@task` is individually tracked — you can see retries, duration, and state in the UI
- `@task(retries=2)` means Prefect will retry the task up to 2 times before marking it `Failed`
- **The flow code is infrastructure-agnostic** — the same `etl_pipeline` function runs identically whether the worker is a subprocess, Docker container, or Kubernetes pod

In [ ]:
import asyncio
from prefect.client.orchestration import get_client
from prefect.client.schemas.actions import DeploymentCreate

DEPLOYMENT_NAME = "etl-daily"

async def register_deployment():
    async with get_client() as client:
        # Register the flow — returns a UUID
        flow_id = await client.create_flow_from_name("etl-pipeline")
        print(f"Flow registered: id={flow_id}")

        dep_id = await client.create_deployment(
            DeploymentCreate(
                name=DEPLOYMENT_NAME,
                flow_id=flow_id,
                version="1.0.0",
                # entrypoint = "path/to/module.py:function_name"
                # Workers use this to import and call the flow
                entrypoint="etl_pipeline:etl_pipeline",
                parameters={
                    "source": "api-v1",
                    "destination": "warehouse",
                    "batch_size": 500,
                    "multiplier": 1.0,
                },
                tags=["etl", "daily", "day-08"],
                work_pool_name=PROCESS_POOL_NAME,  # ← links to our process pool
                description="Daily ETL: extract from API, transform, load to warehouse.",
            )
        )
        print(f"Deployment registered: id={dep_id}")
        print(f"  Name:      {DEPLOYMENT_NAME}")
        print(f"  Pool:      {PROCESS_POOL_NAME}")
        print(f"  Entry:     etl_pipeline:etl_pipeline")
        return dep_id

deployment_id = asyncio.run(register_deployment())
print(f"\nDeployment is now visible in the UI as: etl-pipeline / {DEPLOYMENT_NAME}")
print("CLI: prefect deployment inspect 'etl-pipeline/etl-daily'")

**What just happened?**
- `create_flow_from_name()` registers the flow with the server and returns a UUID; the deployment references that UUID
- `work_pool_name=PROCESS_POOL_NAME` links the deployment to our pool — runs go into the pool queue
- **Nothing has executed** — the deployment is a recipe; workers execute it
- In the Prefect UI: **Deployments** page now shows `etl-pipeline / etl-daily` with a "Quick Run" button

---
## Step 4 · Workers — polling interval and the execution loop

A **worker** is a long-running process that:
1. Connects to the Prefect server
2. Polls its assigned work pool every N seconds (default: 15 s)
3. Fetches any runs in `Scheduled` state that are due
4. Submits them to the appropriate infrastructure (subprocess, container, pod)
5. Reports state changes (Started → Completed / Failed) back to the server

```
Worker polling loop:
  every 15 s:
    GET /api/work_pools/{pool}/get_scheduled_flow_runs
    for each run:
      spawn infrastructure (subprocess / container / pod)
      monitor until terminal state
      PATCH /api/flow_runs/{id}/set_state → Completed | Failed
```

**Polling interval settings:**

| Setting | Default | Description |
|---|---|---|
| `PREFECT_WORKER_QUERY_SECONDS` | `10` | How often the worker polls |
| `PREFECT_WORKER_PREFETCH_SECONDS` | `10` | Fetch runs scheduled up to N sec in the future |
| `--limit` CLI flag | `10` | Max runs to fetch per poll |

Lower polling interval = lower latency but more API calls.

In [ ]:
# Simulate the worker polling loop logic in pure Python
# This shows exactly what happens inside `prefect worker start --pool my-pool`

import time
from dataclasses import dataclass, field
from typing import Optional
from datetime import datetime, timezone

@dataclass
class FakeFlowRun:
    id: str
    name: str
    scheduled_time: datetime
    state: str = "Scheduled"
    parameters: dict = field(default_factory=dict)

class WorkerSimulator:
    """Simplified replica of the Prefect worker polling loop."""

    def __init__(self, pool_name: str, poll_interval: int = 15):
        self.pool_name = pool_name
        self.poll_interval = poll_interval
        self.queue: list[FakeFlowRun] = []
        self._poll_count = 0

    def enqueue(self, run: FakeFlowRun):
        """Server puts runs here when they are scheduled."""
        self.queue.append(run)
        print(f"  [Server] Enqueued run '{run.name}' (id={run.id[:8]}…) → state=Scheduled")

    def poll(self, now: Optional[datetime] = None) -> list[FakeFlowRun]:
        """Fetch runs due now (scheduled_time <= now)."""
        now = now or datetime.now(timezone.utc)
        self._poll_count += 1
        due = [r for r in self.queue if r.scheduled_time <= now and r.state == "Scheduled"]
        print(f"  [Worker] Poll #{self._poll_count}: found {len(due)} due run(s)")
        return due

    def execute(self, run: FakeFlowRun):
        """Launch subprocess / container and wait for completion."""
        run.state = "Running"
        print(f"  [Worker] Launching subprocess for run '{run.name}' params={run.parameters}")
        # In real worker: subprocess.run(["python", "-m", "prefect.engine", run_id])
        time.sleep(0.05)  # simulate brief execution
        run.state = "Completed"
        print(f"  [Worker] Run '{run.name}' → Completed")


# Demonstrate the loop
now = datetime(2024, 1, 15, 8, 0, 0, tzinfo=timezone.utc)

worker = WorkerSimulator(pool_name="my-pool", poll_interval=15)

# Server schedules two runs
print("=== Server scheduling runs ===")
worker.enqueue(FakeFlowRun("run-001", "etl-daily/run-1",
    scheduled_time=datetime(2024, 1, 15, 7, 55, tzinfo=timezone.utc),
    parameters={"source": "api-v1", "batch_size": 500}))
worker.enqueue(FakeFlowRun("run-002", "etl-daily/run-2",
    scheduled_time=datetime(2024, 1, 15, 8, 5, tzinfo=timezone.utc),  # future
    parameters={"source": "api-v2", "batch_size": 100}))

print("\n=== Worker polling (simulated) ===")
# Poll 1 — run-001 is due, run-002 is not yet
due = worker.poll(now=now)
for run in due:
    worker.execute(run)

# Poll 2 — fast-forward 10 minutes
now_plus10 = datetime(2024, 1, 15, 8, 10, tzinfo=timezone.utc)
due = worker.poll(now=now_plus10)
for run in due:
    worker.execute(run)

print("\n=== Final queue state ===")
for run in worker.queue:
    print(f"  {run.name}: {run.state}")

**What just happened?**
- The `WorkerSimulator` shows the **exact polling loop** that `prefect worker start` runs internally
- Run-001 (past due) is picked up on the first poll; Run-002 (future) waits for the next poll after its scheduled time
- **`prefetch_seconds`** — workers actually fetch runs scheduled slightly in the future so they start on time despite poll latency
- In production the worker spawns a real subprocess via `prefect.engine` — the flow code runs in an isolated Python process

---
## Step 5 · Start a worker — CLI reference and configuration

In production you run a worker as a long-lived process — usually managed by `systemd`, Docker, or Kubernetes.

```bash
# Start a worker for the process pool
prefect worker start --pool my-pool

# Worker options
prefect worker start \
  --pool my-pool \
  --name worker-01 \
  --limit 4 \
  --work-queue priority-queue \
  --prefetch-seconds 30
```

**Worker flags:**

| Flag | Default | Effect |
|---|---|---|
| `--pool` | required | Work pool to poll |
| `--name` | auto | Human-readable worker name |
| `--limit` | 10 | Max runs fetched per poll |
| `--work-queue` | default | Poll only a named sub-queue |
| `--prefetch-seconds` | 10 | Fetch runs scheduled N sec ahead |
| `--install-policy` | `prompt` | Auto-install Python dependencies |

**Graceful shutdown:** `Ctrl+C` sends SIGINT — the worker finishes in-flight runs before stopping.

In [ ]:
# Demonstrate worker configuration using environment variables
# These env vars control the worker behaviour without touching CLI flags

import os

WORKER_ENV_SETTINGS = {
    "PREFECT_WORKER_QUERY_SECONDS": "10",          # poll every 10 seconds
    "PREFECT_WORKER_PREFETCH_SECONDS": "15",        # fetch runs up to 15 s in the future
    "PREFECT_WORKER_WEBSERVER_HOST": "0.0.0.0",    # worker health endpoint
    "PREFECT_WORKER_WEBSERVER_PORT": "8080",        # port for health checks
    "PREFECT_API_URL": "http://127.0.0.1:4200/api", # server URL
    "PREFECT_API_KEY": "<your-prefect-cloud-key>",  # for Prefect Cloud
}

print("Worker environment configuration:")
print("-" * 55)
for key, val in WORKER_ENV_SETTINGS.items():
    print(f"  {key:<40} = {val}")

# Describe the worker lifecycle states
lifecycle = [
    ("Starting",   "Worker process boots, connects to server, registers itself"),
    ("Ready",      "Polling loop active — fetching scheduled runs every N seconds"),
    ("Executing",  "Worker has spawned infrastructure for one or more runs"),
    ("Draining",   "Shutdown requested — finishing in-flight runs, accepting no new ones"),
    ("Stopped",    "All runs completed, worker process exits cleanly"),
]

print("\nWorker lifecycle states:")
print("-" * 55)
for state, description in lifecycle:
    print(f"  {state:<12} → {description}")

**What just happened?**
- All `PREFECT_WORKER_*` settings control timing, concurrency, and network configuration
- `PREFECT_WORKER_PREFETCH_SECONDS` is critical for high-frequency schedules — fetch runs slightly early to prevent late starts
- **Health endpoint** (`0.0.0.0:8080`) exposes a `/health` route — use it with Kubernetes liveness probes
- In production: add the worker to a `systemd` unit or Kubernetes Deployment so it auto-restarts on failure

---
## Step 6 · Docker work pool — configuration and image requirements

The `docker` pool type runs each flow run in an isolated Docker container.
This solves three common production problems:

| Problem | How Docker pool solves it |
|---|---|
| Dependency conflicts | Each run gets a fresh container with pinned dependencies |
| Reproducibility | Image tag = exact Python + package versions |
| Resource isolation | CPU and memory limits per container |
| Cleanup | Containers are removed after the run completes |

**Base job template** fields for Docker pools:

| Field | Example | Description |
|---|---|---|
| `image` | `myrepo/my-flow:1.2.3` | Container image to run |
| `image_pull_policy` | `IfNotPresent` | When to pull from registry |
| `network_mode` | `bridge` | Docker network |
| `env` | `{"ENV": "prod"}` | Additional environment variables |
| `auto_remove` | `true` | Remove container after run |
| `volumes` | `["/data:/data"]` | Mount host paths |
| `mem_limit` | `"512m"` | Memory cap per container |

In [ ]:
import json

# Construct a Docker work pool configuration (base job template)
# This is what the server stores and the Docker worker reads on each run

docker_base_job_template = {
    "job_configuration": {
        "image": "{{image}}",          # placeholder — filled per deployment
        "image_pull_policy": "IfNotPresent",
        "auto_remove": True,            # clean up container after run
        "network_mode": "bridge",
        "stream_output": True,          # pipe container stdout to Prefect logs
        "env": {
            "PREFECT_API_URL": "http://host.docker.internal:4200/api",
            "ENVIRONMENT": "production",
        },
        "labels": {
            "managed-by": "prefect",
            "pool": "docker-pool",
        },
    },
    "variables": {
        "type": "object",
        "properties": {
            "image": {
                "type": "string",
                "description": "Docker image for the flow run",
                "default": "prefecthq/prefect:2-python3.10",
            }
        },
    },
}

print("Docker work pool — base_job_template:")
print(json.dumps(docker_base_job_template, indent=2))

print("\n--- Production CLI to create the Docker pool ---")
print("prefect work-pool create --type docker docker-pool")
print("")
print("--- Then deploy a flow to the Docker pool ---")
print("prefect deploy etl_pipeline.py:etl_pipeline \\")
print("  --name etl-docker \\")
print("  --pool docker-pool \\")
print("  --variable image=myrepo/etl-flow:1.2.3")

**What just happened?**
- The `base_job_template` is a JSON schema stored in the work pool — every deployment targeting this pool inherits these defaults
- `{{image}}` is a **template variable** — each deployment can override it with its own image tag
- `"PREFECT_API_URL": "http://host.docker.internal:4200/api"` is the correct URL when running Prefect server on the host and containers on the same Docker network
- **`stream_output: true`** pipes the container's stdout directly to Prefect's log store — no separate logging setup needed

---
## Step 7 · Build and push a Docker image for a flow

A minimal `Dockerfile` for a Prefect flow:

```dockerfile
FROM prefecthq/prefect:2-python3.10

# Install your flow's dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy flow code into the image
COPY flows/ /opt/prefect/flows/

# Prefect will call: python -m prefect.engine <run_id>
# which imports the flow via the entrypoint stored in the deployment
```

**`prefect deploy` with `--build`** builds and pushes the image automatically:

```bash
prefect deploy flows/etl_pipeline.py:etl_pipeline \
  --name etl-docker \
  --pool docker-pool \
  --build \
  --push \
  --variable image=myrepo/etl-flow:$(git rev-parse --short HEAD)
```

In [ ]:
import textwrap
import tempfile
import os

# Generate a production-ready Dockerfile for a Prefect flow
# We write it to disk so you can see exactly what it contains

DOCKERFILE_CONTENT = textwrap.dedent("""\
    # Prefect base image with Python 3.10
    FROM prefecthq/prefect:2-python3.10

    # Set working directory
    WORKDIR /opt/prefect/flows

    # Copy and install dependencies first (layer cache optimisation)
    COPY requirements.txt .
    RUN pip install --no-cache-dir -r requirements.txt

    # Copy flow code
    COPY flows/ .

    # Prefect runs this when the container starts:
    # python -m prefect.engine <flow_run_id>
    # No CMD/ENTRYPOINT needed — Prefect's worker passes the run id at runtime
""")

REQUIREMENTS_CONTENT = textwrap.dedent("""\
    prefect>=2.14
    pandas>=2.0
    sqlalchemy>=2.0
    # add your project dependencies here
""")

# Write files to a temp directory to demonstrate the structure
tmp = tempfile.mkdtemp()

dockerfile_path = os.path.join(tmp, "Dockerfile")
requirements_path = os.path.join(tmp, "requirements.txt")

with open(dockerfile_path, "w") as f:
    f.write(DOCKERFILE_CONTENT)
with open(requirements_path, "w") as f:
    f.write(REQUIREMENTS_CONTENT)

print(f"Files written to: {tmp}")
print("\n=== Dockerfile ===")
print(DOCKERFILE_CONTENT)
print("=== requirements.txt ===")
print(REQUIREMENTS_CONTENT)

print("\n--- Build and push commands ---")
print("docker build -t myrepo/etl-flow:1.0.0 .")
print("docker push myrepo/etl-flow:1.0.0")
print("")
print("--- Or let Prefect do it automatically ---")
print("prefect deploy flows/etl_pipeline.py:etl_pipeline --build --push --pool docker-pool")

**What just happened?**
- `prefecthq/prefect:2-python3.10` is the official base image — it includes the Prefect agent and Python runtime
- Copying `requirements.txt` before code takes advantage of **Docker layer caching** — dependencies rebuild only when requirements change
- **No `CMD` or `ENTRYPOINT`** — the Docker worker injects `python -m prefect.engine <run_id>` at container start
- `prefect deploy --build --push` automates the build+push+register cycle — ideal for CI/CD pipelines

---
## Step 8 · Verify a worker picks up a run — full end-to-end trace

When a worker picks up a run, the state machine moves:

```
Scheduled → Running → Completed
                   ↘ Failed (on unhandled exception)
                   ↘ Crashed (on infra failure)
```

The key difference between **Failed** and **Crashed**:

| State | Cause | Retry behaviour |
|---|---|---|
| `Failed` | Flow raised an exception | Respects `@flow(retries=N)` |
| `Crashed` | Worker lost contact / infra died | Server moves to `Crashed`, no retry |
| `Cancelling` | User cancelled via UI/CLI | Worker kills subprocess / container |

In [ ]:
from prefect import flow, task
from prefect.testing.utilities import prefect_test_harness
from prefect.states import Completed, Failed
import traceback

# Demonstrate state transitions: Scheduled → Running → Completed / Failed

@task
def validate_input(value: int) -> int:
    """Task that fails on negative input — demonstrates retry behaviour."""
    if value < 0:
        raise ValueError(f"Input must be >= 0, got {value}")
    return value * 10

@flow(name="state-demo", log_prints=True, retries=1, retry_delay_seconds=1)
def state_demo_flow(value: int = 5) -> dict:
    """Flow that demonstrates success and failure state transitions."""
    result = validate_input(value=value)
    return {"input": value, "output": result}


print("=== Scenario 1: successful run (Scheduled → Running → Completed) ===")
with prefect_test_harness():
    state = state_demo_flow(value=7, return_state=True)
    print(f"  Final state: {state.type}")
    if state.is_completed():
        print(f"  Result:      {state.result()}")

print()
print("=== Scenario 2: failing run (Scheduled → Running → Failed) ===")
with prefect_test_harness():
    state = state_demo_flow(value=-3, return_state=True)
    print(f"  Final state: {state.type}")
    if state.is_failed():
        print(f"  The flow failed (as expected for negative input)")

print()
print("In the UI: Flow Runs shows both runs with their state badges")
print("CLI: prefect flow-run ls --state FAILED")

**What just happened?**
- `return_state=True` returns a Prefect `State` object instead of the raw return value — lets you inspect state in tests
- `state.is_completed()` / `state.is_failed()` are the idiomatic checks — do not compare `.type` strings directly
- **Flow-level retries** (`@flow(retries=1)`) trigger on any unhandled exception in the flow body
- In the real worker: the subprocess exit code determines whether the run moves to `Completed` or `Crashed`

---
## Challenge

You have the following data processing flow:

In [ ]:
# Challenge: Work Pools and Workers
#
# Given this flow:
from prefect import flow, task
from prefect.testing.utilities import prefect_test_harness
from prefect.client.orchestration import get_client
from prefect.client.schemas.actions import WorkPoolCreate, DeploymentCreate
import asyncio

@task(retries=1, retry_delay_seconds=2)
def ingest(source: str, rows: int) -> list[dict]:
    return [{"id": i, "src": source, "v": i ** 2} for i in range(1, rows + 1)]

@task
def summarise(records: list[dict]) -> dict:
    return {"count": len(records), "total": sum(r["v"] for r in records)}

@flow(name="data-ingest", log_prints=True)
def ingest_flow(source: str = "default", rows: int = 10) -> dict:
    data = ingest(source=source, rows=rows)
    result = summarise(records=data)
    print(f"Summary: {result}")
    return result

# Task 1: Run the flow locally with prefect_test_harness() using default params
# YOUR CODE HERE

# Task 2: Run it again with source="warehouse", rows=20
# YOUR CODE HERE

# Task 3: Print which run produced a higher total
# YOUR CODE HERE

# Task 4: Using get_client(), create a work pool named "challenge-pool"
#         with type="process" and then register a deployment named
#         "ingest-daily" that targets it.
#         Hint: use asyncio.run() and async with get_client() as client
# YOUR CODE HERE

---
## Day 8 key concepts recap

| Concept | What to remember |
|---|---|
| Work pool | Server-side queue; deployments target a pool; workers poll it |
| Worker | Long-running process that polls a pool and spawns infrastructure |
| Process pool | Runs flows as local subprocesses — best for development |
| Docker pool | Runs each flow run in an isolated container — reproducible environments |
| Polling interval | `PREFECT_WORKER_QUERY_SECONDS` (default 10 s) — lower = less latency, more API calls |
| Prefetch seconds | Fetch runs a few seconds before their scheduled time to avoid late starts |
| State machine | Scheduled → Running → Completed / Failed / Crashed |
| base_job_template | JSON schema of infra settings stored in the pool; deployments can override fields |
| `entrypoint` | `module.py:function` — tells the worker what to import inside the container/subprocess |

> **Tip:** Work pools decouple scheduling from execution — your Prefect server schedules runs, but workers running anywhere (laptop, VM, K8s pod) do the actual work.

---
## What's next
**Day 9** → Schedules, Automations, and Event-Driven Triggers: add CronSchedule and IntervalSchedule to deployments, build automations that chain flows on failure events, and create webhook-triggered runs.

Mark Day 8 complete in your [tracker](../index.html).